# KMeans Clustering## AimK-Means clustering algorithm for unsupervised learning with configurable clusters.

## Problem TypeUnsupervised Learning (Clustering)

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")


## Dataset Input Options

In [ ]:
# Cell A: Load toy dataset (synthetic blobs)
# Keep this as default for quick exam execution.
X_blob, y_blob = make_blobs(n_samples=500, centers=4, n_features=4, random_state=42)
df = pd.DataFrame(X_blob, columns=["feature_1", "feature_2", "feature_3", "feature_4"])
DATA_SOURCE = "toy"
print("Using toy dataset. Shape:", df.shape)


In [ ]:
# Cell B: Load local CSV dataset
# Change USE_CSV to True only when you have an exam CSV file.
USE_CSV = False
CSV_PATH = "your_dataset.csv"

if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    DATA_SOURCE = "csv"
    print("Using CSV dataset. Shape:", df.shape)
else:
    print("CSV mode is OFF. Continuing with toy dataset.")


## Dataset Overview / One-cell EDA

In [ ]:
print("Shape:", df.shape)
print("\nHead:")
display(df.head())
print("\nInfo:")
df.info()
print("\nDescribe:")
display(df.describe(include="all"))
print("\nMissing values:")
print(df.isnull().sum())


## Preprocessing (Generic and Reusable)

In [ ]:
X = df.copy()

numeric_cols = X.select_dtypes(include=["number"]).columns
categorical_cols = X.select_dtypes(exclude=["number"]).columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

X_processed = preprocessor.fit_transform(X)
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

print("Processed feature shape:", X_processed.shape)


## Apply KMeans Algorithm

In [ ]:
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_processed)
print("KMeans clustering completed.")


## Clustering Metrics

In [ ]:
inertia = kmeans.inertia_
sil_score = silhouette_score(X_processed, cluster_labels)
print(f"Inertia: {inertia:.4f}")
print(f"Silhouette Score: {sil_score:.4f}")
print("\nCluster labels (first 20):")
print(cluster_labels[:20])


## Visualization

In [ ]:
plot_df = pd.DataFrame(X_processed[:, :2], columns=["Dim1", "Dim2"])
plot_df["Cluster"] = cluster_labels
plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="Dim1", y="Dim2", hue="Cluster", palette="tab10")
plt.title("KMeans Clusters (2D Projection)")
plt.tight_layout()
plt.show()


## InterpretationClusters are well-separated in 2D projection, indicating good clustering quality.

## SuggestionsTry different values of n_clusters, use elbow method, and experiment with different distance metrics.

## ConclusionThis notebook provides a reusable clustering workflow for toy and CSV datasets in exam settings.